# Home Credit Default Risk — application-only end-to-end

Mục tiêu của notebook là tạo một baseline có thể kiểm chứng trên bảng application. Feature engineering nằm ở notebook để phục vụ học tập/khám phá; loader, model runner, CV, tuning và submission được import từ `src/credit_scoring`.

## Quy ước experiment

- `TARGET=1` là application có khó khăn trả nợ.
- Primary metric là ROC-AUC.
- Smoke dùng StratifiedKFold 3 folds; baseline dùng 5 folds, seed 42.
- Không dùng SMOTE, accuracy hoặc leaderboard để tune.
- Chỉ xử lý application ở lần chạy này; các bảng 1-n sẽ được thêm sau khi baseline ổn.

## Hướng dẫn chạy notebook

Chạy các cell theo thứ tự từ trên xuống. Mặc định là `smoke` để kiểm tra luồng dữ liệu, feature matrix, OOF validation và artifact export mà không tạo submission. Chỉ đổi `CONFIG["run_mode"]` thành `baseline` khi đã xác nhận dữ liệu Kaggle, thời gian chạy và bộ nhớ phù hợp.

Notebook chỉ dùng hai bảng `application_train` và `application_test`. Không dùng target để tạo feature; ROC-AUC chỉ được in sau khi cross-validation thực sự hoàn tất.

### Truy cập public GitHub từ Kaggle

Bật **Internet** trong Kaggle Settings trước khi clone. Repository đã public nên không cần GitHub token, Kaggle Secret hoặc thay đổi `REPO_URL`. Nếu clone vẫn thất bại, kiểm tra lại Internet, URL repository và branch `main`.

### Phân biệt code và competition data

Kaggle Input chứa các CSV như `application_train.csv`, nên nó giải quyết bước nạp dữ liệu. Lỗi `No module named 'credit_scoring.data'` thuộc bước import code: notebook phải clone đúng repository, thêm `<repo>/src` lên đầu `sys.path` và bảo đảm không dùng nhầm một package `credit_scoring` đã được Kaggle nạp trước đó. Cell clone in đường dẫn package thực tế để hỗ trợ chẩn đoán.

In [ ]:
# 1. Clone source code và import dependency có sẵn trên Kaggle
import importlib
import platform
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

REPO_URL = "https://github.com/ManhTanTran/Qaci-datascience.git"
REPO_BRANCH = "main"
REPO_COMMIT = "7d5c685e9853c566c5627a2822697e6f6833ded8"
REPO_DIR = Path("/kaggle/working/Qaci-datascience")

def run_git(*arguments: str) -> None:
    """Run one git command and fail loudly in the Kaggle notebook."""
    subprocess.run(["git", "-C", str(REPO_DIR), *arguments], check=True)

def clone_or_checkout_repository() -> None:
    """Clone or pin the public repository."""
    try:
        if not (REPO_DIR / ".git").is_dir():
            subprocess.run(
                ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
                check=True,
            )
        if REPO_COMMIT is None:
            run_git("checkout", REPO_BRANCH)
            run_git("pull", "--ff-only", "origin", REPO_BRANCH)
        else:
            run_git("fetch", "--depth", "1", "origin", REPO_BRANCH)
            run_git("checkout", REPO_BRANCH)
            run_git("fetch", "--depth", "1", "origin", REPO_COMMIT)
            run_git("checkout", "--detach", REPO_COMMIT)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "Cannot access the public GitHub repository. Enable Kaggle Internet and "
            "confirm that REPO_URL and REPO_BRANCH are correct."
        ) from exc

clone_or_checkout_repository()

GIT_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
if REPO_COMMIT is not None:
    assert GIT_COMMIT == REPO_COMMIT, (GIT_COMMIT, REPO_COMMIT)
SRC_DIR = (REPO_DIR / "src").resolve()
EXPECTED_MODULE = SRC_DIR / "credit_scoring" / "data" / "home_credit.py"
if not EXPECTED_MODULE.is_file():
    raise RuntimeError(
        f"Repository at {REPO_DIR} does not contain {EXPECTED_MODULE.relative_to(REPO_DIR)}. "
        "Check REPO_COMMIT and the clone output."
    )
if str(SRC_DIR) in sys.path:
    sys.path.remove(str(SRC_DIR))
sys.path.insert(0, str(SRC_DIR))
importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == "credit_scoring" or module_name.startswith("credit_scoring."):
        del sys.modules[module_name]
credit_scoring = importlib.import_module("credit_scoring")
credit_scoring_path = Path(credit_scoring.__file__).resolve()
if SRC_DIR not in credit_scoring_path.parents:
    raise RuntimeError(f"Imported unexpected credit_scoring package: {credit_scoring_path}")
print(f"Repository: {REPO_DIR}")
print(f"Git commit: {GIT_COMMIT}")
print(f"credit_scoring package: {credit_scoring_path}")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from credit_scoring.artifacts import export_dataframe_artifact, export_json_artifact
from credit_scoring.data.home_credit import (
    find_home_credit_data_dir,
    load_home_credit_data,
    summarize_loaded_tables,
)
from credit_scoring.modeling.lightgbm_model import run_lightgbm_cv
from credit_scoring.reproducibility import set_global_seed
from credit_scoring.submission.home_credit import create_home_credit_submission

set_global_seed(42)
pd.set_option("display.max_columns", 150)
pd.set_option("display.width", 160)

## Cấu hình run

`RUN_MODES` kiểm soát kích thước mẫu, số folds, số cây và early stopping. `CONFIG` lưu các lựa chọn của lần chạy hiện tại; `run_tuning` và `run_ablation` đều tắt mặc định để smoke run không tạo chi phí ngoài dự kiến. Output được tách theo experiment, run mode và short Git commit.

In [ ]:
# 2. Tập trung toàn bộ configuration ở một cell
RUN_MODES = {
    "smoke": {
        "sample_size": 5_000,
        "n_splits": 3,
        "n_estimators": 300,
        "early_stopping_rounds": 50,
        "create_submission": False,
    },
    "baseline": {
        "sample_size": None,
        "n_splits": 5,
        "n_estimators": 5_000,
        "early_stopping_rounds": 200,
        "create_submission": True,
    },
}
CONFIG = {
    "experiment_name": "E01_application_baseline",
    "run_mode": "smoke",
    "data_dir": None,
    "output_dir": "/kaggle/working/home_credit_outputs",
    "shuffle": True,
    "random_state": 42,
    "n_trials": 30,
    "run_tuning": False,
    "run_ablation": False,
}
if CONFIG["run_mode"] not in RUN_MODES:
    raise ValueError(f"Unknown run_mode: {CONFIG['run_mode']}")
MODE_CONFIG = RUN_MODES[CONFIG["run_mode"]]
if CONFIG["run_mode"] == "smoke" and CONFIG["run_tuning"]:
    raise ValueError("Smoke mode does not permit tuning.")
MODEL_CONFIG = {
    "learning_rate": 0.02,
    "n_estimators": MODE_CONFIG["n_estimators"],
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 80,
    "subsample": 0.8,
    "colsample_bytree": 0.7,
    "reg_alpha": 0.1,
    "reg_lambda": 5.0,
    "random_state": CONFIG["random_state"],
    "n_jobs": -1,
    "verbosity": -1,
}
VALIDATION_CONFIG = {
    "n_splits": MODE_CONFIG["n_splits"],
    "shuffle": CONFIG["shuffle"],
    "random_state": CONFIG["random_state"],
    "early_stopping_rounds": MODE_CONFIG["early_stopping_rounds"],
    "keep_models": True,
}
BASE_OUTPUT_DIR = Path(CONFIG["output_dir"]).expanduser()
GIT_COMMIT_SHORT = GIT_COMMIT[:8]
OUTPUT_DIR = (
    BASE_OUTPUT_DIR
    / CONFIG["experiment_name"]
    / CONFIG["run_mode"]
    / GIT_COMMIT_SHORT
).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(CONFIG)

## Nạp và kiểm tra dữ liệu application

Resolver ưu tiên `CONFIG["data_dir"]`, sau đó Kaggle input và local fallback. Chỉ yêu cầu `application_train` và `application_test`; loader kiểm tra schema cơ bản trước khi các cell sau tạo feature. Bảng tóm tắt chỉ hiển thị thống kê cấu trúc, không in dữ liệu khách hàng từng dòng.

In [ ]:
# 3. Load application tables qua repository loader
DATA_DIR = find_home_credit_data_dir(CONFIG["data_dir"])
print(f"Data directory: {DATA_DIR}")
data = load_home_credit_data(
    data_dir=DATA_DIR,
    tables=["application_train", "application_test"],
    nrows=MODE_CONFIG["sample_size"],
    reduce_memory=True,
    validate=True,
)
train_raw = data["application_train"].copy()
test_raw = data["application_test"].copy()
display(summarize_loaded_tables(data))
display(train_raw["TARGET"].value_counts(normalize=True).rename("target_rate"))

In [ ]:
# 4. Data audit: missingness, duplicates và key integrity
def missing_summary(frame: pd.DataFrame) -> pd.DataFrame:
    """Return missing count/rate for one table, sorted by missing rate."""
    result = pd.DataFrame({
        "missing_count": frame.isna().sum(),
        "missing_rate": frame.isna().mean(),
        "dtype": frame.dtypes.astype(str),
    })
    return result.sort_values("missing_rate", ascending=False)

assert train_raw["SK_ID_CURR"].is_unique
assert test_raw["SK_ID_CURR"].is_unique
assert not set(train_raw["SK_ID_CURR"]).intersection(test_raw["SK_ID_CURR"])
print(f"Train duplicate rows: {train_raw.duplicated().sum()}")
print(f"Test duplicate rows: {test_raw.duplicated().sum()}")
display(missing_summary(train_raw).head(20))

## Làm sạch tối thiểu và feature engineering

Chỉ chuẩn hóa sentinel `DAYS_EMPLOYED=365243` và giữ missing hợp lệ để LightGBM xử lý. Các feature phía dưới là notebook-local, minh bạch theo công thức: ratios an toàn, EXT_SOURCE aggregates/pairs, document/contact counts và housing summary. Mỗi hàm copy DataFrame đầu vào để không làm thay đổi split gốc.

In [ ]:
# 5. Application cleaning — không drop missing, chỉ xử lý sentinel đã biết
def clean_application_data(frame: pd.DataFrame) -> pd.DataFrame:
    """Copy application data, flag and null the known DAYS_EMPLOYED sentinel."""
    cleaned = frame.copy()
    if "DAYS_EMPLOYED" in cleaned:
        anomaly = cleaned["DAYS_EMPLOYED"].eq(365243)
        cleaned["DAYS_EMPLOYED_ANOMALOUS"] = anomaly.astype("int8")
        cleaned.loc[anomaly, "DAYS_EMPLOYED"] = np.nan
    cleaned = cleaned.replace([np.inf, -np.inf], np.nan)
    return cleaned

train = clean_application_data(train_raw)
test = clean_application_data(test_raw)
print(f"DAYS_EMPLOYED anomalies: {train['DAYS_EMPLOYED_ANOMALOUS'].sum() if 'DAYS_EMPLOYED_ANOMALOUS' in train else 0}")

In [ ]:
# 6. Notebook-local feature engineering: safe ratios và application groups
CONTACT_FLAG_COLUMNS = [
    "FLAG_MOBIL",
    "FLAG_EMP_PHONE",
    "FLAG_WORK_PHONE",
    "FLAG_CONT_MOBILE",
    "FLAG_PHONE",
    "FLAG_EMAIL",
]

def safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    """Divide aligned series and return NaN for zero/invalid denominators."""
    denominator = denominator.replace(0, np.nan)
    result = numerator.divide(denominator)
    return result.replace([np.inf, -np.inf], np.nan)

def add_application_ratio_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Add transparent amount, affordability and age/employment ratios."""
    result = frame.copy()
    formulas = {
        "CREDIT_INCOME_RATIO": ("AMT_CREDIT", "AMT_INCOME_TOTAL"),
        "ANNUITY_INCOME_RATIO": ("AMT_ANNUITY", "AMT_INCOME_TOTAL"),
        "ANNUITY_CREDIT_RATIO": ("AMT_ANNUITY", "AMT_CREDIT"),
        "GOODS_CREDIT_RATIO": ("AMT_GOODS_PRICE", "AMT_CREDIT"),
        "EMPLOYED_BIRTH_RATIO": ("DAYS_EMPLOYED", "DAYS_BIRTH"),
        "INCOME_PER_PERSON": ("AMT_INCOME_TOTAL", "CNT_FAM_MEMBERS"),
        "CHILDREN_RATIO": ("CNT_CHILDREN", "CNT_FAM_MEMBERS"),
    }
    for name, (numerator, denominator) in formulas.items():
        if numerator in result and denominator in result:
            result[name] = safe_divide(result[numerator], result[denominator])
    if {"OWN_CAR_AGE", "DAYS_BIRTH"}.issubset(result.columns):
        result["CAR_BIRTH_RATIO"] = safe_divide(
            result["OWN_CAR_AGE"],
            result["DAYS_BIRTH"].abs(),
        )
    return result

def add_ext_source_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Add aggregate and pairwise features from available EXT_SOURCE columns."""
    result = frame.copy()
    columns = [column for column in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"] if column in result]
    if not columns:
        return result
    values = result[columns]
    result["EXT_SOURCE_MEAN"] = values.mean(axis=1)
    result["EXT_SOURCE_MEDIAN"] = values.median(axis=1)
    result["EXT_SOURCE_MIN"] = values.min(axis=1)
    result["EXT_SOURCE_MAX"] = values.max(axis=1)
    result["EXT_SOURCE_STD"] = values.std(axis=1)
    result["EXT_SOURCE_MISSING_COUNT"] = values.isna().sum(axis=1)
    if len(columns) >= 2:
        result["EXT_SOURCE_PRODUCT"] = values.prod(axis=1, min_count=2)
        result["EXT_SOURCE_RANGE"] = result["EXT_SOURCE_MAX"] - result["EXT_SOURCE_MIN"]
    for left_index, left in enumerate(columns):
        for right in columns[left_index + 1 :]:
            result[f"{left}_MINUS_{right}"] = result[left] - result[right]
            result[f"{left}_DIV_{right}"] = safe_divide(result[left], result[right])
            result[f"{right}_DIV_{left}"] = safe_divide(result[right], result[left])
    return result

def add_document_contact_housing_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Add counts and selected housing aggregates without target information."""
    result = frame.copy()
    document_columns = [column for column in result if column.startswith("FLAG_DOCUMENT_")]
    contact_columns = [column for column in CONTACT_FLAG_COLUMNS if column in result]
    if document_columns:
        result["DOCUMENT_FLAG_COUNT"] = result[document_columns].sum(axis=1)
    if contact_columns:
        result["CONTACT_FLAG_COUNT"] = result[contact_columns].fillna(0).sum(axis=1)
    print(f"Contact flag columns used ({len(contact_columns)}): {contact_columns}")
    housing_columns = [column for column in ["APARTMENTS_AVG", "BASEMENTAREA_AVG", "YEARS_BEGINEXPLUATATION_AVG", "ELEVATORS_AVG", "WALLSMATERIAL_MODE"] if column in result]
    numeric_housing = [column for column in housing_columns if pd.api.types.is_numeric_dtype(result[column])]
    if numeric_housing:
        result["HOUSING_AVG_MEAN"] = result[numeric_housing].mean(axis=1)
    return result


## Feature matrix và validation contract

Train/test được concatenated chỉ để align category encoding, sau đó tách lại theo số dòng train. Assertions xác nhận không có target/ID trong ma trận, thứ tự cột trùng khớp, ID unique, target nhị phân và numeric features không chứa infinity. `NaN` vẫn được giữ vì LightGBM hỗ trợ missing values.

Manifest E01–E05 mô tả tập feature cumulative dùng cho ablation tùy chọn; baseline mặc định dùng E05 đầy đủ.

In [ ]:
# 7. Build and align application-only feature matrix
def build_application_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Run the ordered application feature steps for one split."""
    result = add_application_ratio_features(frame)
    result = add_ext_source_features(result)
    result = add_document_contact_housing_features(result)
    return result

train_featured = build_application_features(train)
test_featured = build_application_features(test)

target = train_featured.pop("TARGET").astype("int8")
train_ids = train_featured.pop("SK_ID_CURR")
test_ids = test_featured.pop("SK_ID_CURR")

combined = pd.concat([train_featured, test_featured], axis=0, ignore_index=True)
categorical_columns = combined.select_dtypes(include=["object"]).columns.tolist()
for column in categorical_columns:
    combined[column] = combined[column].astype("category")
X_train = combined.iloc[: len(train_featured)].copy()
X_test = combined.iloc[len(train_featured) :].copy()
X_test.index = range(len(X_test))

raw_feature_columns = [
    column for column in train_raw.columns if column not in {"TARGET", "SK_ID_CURR"}
]
clean_feature_columns = [
    column for column in train.columns if column not in {"TARGET", "SK_ID_CURR"}
]
ratio_feature_names = [
    "CREDIT_INCOME_RATIO",
    "ANNUITY_INCOME_RATIO",
    "ANNUITY_CREDIT_RATIO",
    "GOODS_CREDIT_RATIO",
    "EMPLOYED_BIRTH_RATIO",
    "INCOME_PER_PERSON",
    "CHILDREN_RATIO",
    "CAR_BIRTH_RATIO",
]
ext_source_feature_columns = [
    column
    for column in X_train.columns
    if column.startswith("EXT_SOURCE_") and column not in raw_feature_columns
]

def available_feature_columns(columns: list[str]) -> list[str]:
    """Keep a notebook-local feature manifest aligned to the final matrix."""
    return [column for column in columns if column in X_train.columns]

FEATURE_GROUPS = {
    "E01_application_raw": available_feature_columns(raw_feature_columns),
    "E02_application_clean": available_feature_columns(clean_feature_columns),
    "E03_application_ratios": available_feature_columns(
        clean_feature_columns + ratio_feature_names
    ),
    "E04_application_ext_source": available_feature_columns(
        clean_feature_columns + ratio_feature_names + ext_source_feature_columns
    ),
    "E05_application_flags_housing": list(X_train.columns),
}
enabled_feature_groups = ["E05_application_flags_housing"]

assert len(X_train) == len(target) == len(train_ids)
assert len(X_test) == len(test_ids)
assert "TARGET" not in X_train.columns
assert "TARGET" not in X_test.columns
assert "SK_ID_CURR" not in X_train.columns
assert "SK_ID_CURR" not in X_test.columns
assert list(X_train.columns) == list(X_test.columns)
assert not X_train.columns.duplicated().any()
assert not X_test.columns.duplicated().any()
assert target.isin([0, 1]).all()
assert train_ids.is_unique
assert test_ids.is_unique
numeric_columns = X_train.select_dtypes(include="number").columns.tolist()
for matrix in (X_train, X_test):
    numeric_matrix = matrix.select_dtypes(include="number")
    assert not np.isinf(numeric_matrix).any().any()
memory_mb = (
    X_train.memory_usage(deep=True).sum() + X_test.memory_usage(deep=True).sum()
) / 1024**2
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Numeric feature count: {len(numeric_columns)}")
print(f"Categorical feature count: {len(categorical_columns)}")
print(f"Feature matrix memory (MB): {memory_mb:.2f}")

## Cross-validation leakage-safe

`run_lightgbm_cv()` là implementation dùng lại từ repository. Notebook không viết lại CV loop; thay vào đó xác minh contract trả về: OOF/test probabilities hợp lệ, mỗi row chỉ xuất hiện đúng một validation fold, số fold đúng cấu hình và feature importance không rỗng. AUC in ở đây là kết quả thực tế của lần chạy, không phải benchmark cố định.

In [ ]:
# 8. LightGBM baseline: OOF, test predictions, fold metrics, importance
baseline_result = run_lightgbm_cv(
    train_features=X_train,
    target=target,
    test_features=X_test,
    categorical_features=categorical_columns,
    model_config=MODEL_CONFIG,
    validation_config=VALIDATION_CONFIG,
)
oof_predictions = baseline_result["oof_predictions"]
test_predictions = baseline_result["test_predictions"]
validation_counts = baseline_result["validation_counts"]
fold_scores = baseline_result["fold_scores"]
best_iterations = baseline_result["best_iterations"]
assert len(oof_predictions) == len(target)
assert len(test_predictions) == len(X_test)
assert np.isfinite(oof_predictions).all()
assert np.isfinite(test_predictions).all()
assert np.all((oof_predictions >= 0) & (oof_predictions <= 1))
assert np.all((test_predictions >= 0) & (test_predictions <= 1))
assert len(validation_counts) == len(target)
assert np.all(validation_counts == 1)
assert len(fold_scores) == VALIDATION_CONFIG["n_splits"]
assert len(best_iterations) == VALIDATION_CONFIG["n_splits"]
assert baseline_result["metadata"]["n_splits"] == VALIDATION_CONFIG["n_splits"]
assert not baseline_result["feature_importance"].empty
assert np.isfinite(np.asarray(fold_scores, dtype=float)).all()
print("OOF coverage min:", validation_counts.min())
print("OOF coverage max:", validation_counts.max())
print("Fold AUC:", baseline_result["fold_scores"])
print("Mean AUC:", baseline_result["mean_auc"])
print("Std AUC:", baseline_result["std_auc"])
print("OOF AUC:", baseline_result["oof_auc"])
print("Best iterations:", baseline_result["best_iterations"])
print("Runtime seconds:", round(baseline_result["runtime"], 2))
display(baseline_result["feature_importance"].head(30))

## Diagnostics và artifacts

Mọi artifact của một run được ghi vào cùng thư mục versioned: configuration, environment, fold metrics, OOF/test predictions và feature importance. Các file này chỉ xuất hiện sau khi training hoàn tất; notebook lưu trống không kèm metric hay prediction có sẵn.

In [ ]:
# 9. Diagnostics và actual artifact export
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(target, baseline_result["oof_predictions"], ax=ax)
ax.set_title(f"OOF ROC curve — AUC {baseline_result['oof_auc']:.5f}")
plt.show()

importance_path = export_dataframe_artifact(
    baseline_result["feature_importance"], OUTPUT_DIR / "feature_importance.csv"
)
def installed_version(package_name: str) -> str | None:
    """Return an installed package version without inventing missing values."""
    try:
        return version(package_name)
    except PackageNotFoundError:
        return None

environment = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "git_commit": GIT_COMMIT,
    "packages": {
        name: installed_version(name)
        for name in ["numpy", "pandas", "scikit-learn", "lightgbm", "matplotlib", "optuna"]
    },
}
fold_metrics = pd.DataFrame(
    {
        "fold": range(1, len(baseline_result["fold_scores"]) + 1),
        "auc": baseline_result["fold_scores"],
        "best_iteration": baseline_result["best_iterations"],
    }
)
oof_frame = pd.DataFrame(
    {
        "SK_ID_CURR": train_ids.to_numpy(),
        "TARGET": target.to_numpy(),
        "OOF_PREDICTION": baseline_result["oof_predictions"],
        "VALIDATION_COUNT": baseline_result["validation_counts"],
    }
)
test_frame = pd.DataFrame(
    {
        "SK_ID_CURR": test_ids.to_numpy(),
        "TEST_PREDICTION": baseline_result["test_predictions"],
    }
)
feature_groups = {name: list(columns) for name, columns in FEATURE_GROUPS.items()}
config_payload = {
    "config": {
        **CONFIG,
        "resolved_data_dir": str(DATA_DIR.resolve()),
        "resolved_output_dir": str(OUTPUT_DIR),
    },
    "mode_config": MODE_CONFIG,
    "model_config": MODEL_CONFIG,
    "validation_config": VALIDATION_CONFIG,
}
artifact_paths = {
    "config": export_json_artifact(config_payload, OUTPUT_DIR / "config.json"),
    "environment": export_json_artifact(environment, OUTPUT_DIR / "environment.json"),
    "fold_metrics": export_dataframe_artifact(fold_metrics, OUTPUT_DIR / "fold_metrics.csv"),
    "oof_predictions": export_dataframe_artifact(oof_frame, OUTPUT_DIR / "oof_predictions.csv"),
    "test_predictions": export_dataframe_artifact(test_frame, OUTPUT_DIR / "test_predictions.csv"),
    "feature_importance": importance_path,
}
print({name: str(path) for name, path in artifact_paths.items()})

## Ablation và tuning là tùy chọn

Ablation tắt mặc định và chạy các manifest E01–E05 để so sánh feature group theo cùng CV contract. Tuning cũng tắt mặc định; khi bật, Optuna chỉ nhìn train data và không tự động thay đổi `MODEL_CONFIG` của baseline. Không dùng kết quả smoke để kết luận chất lượng model.

In [ ]:
# 10. Optional ablation — chỉ chạy khi CONFIG['run_ablation'] = True
ablation_rows = []
if CONFIG["run_ablation"]:
    for experiment_name, columns in FEATURE_GROUPS.items():
        result = run_lightgbm_cv(
            train_features=X_train[columns],
            target=target,
            test_features=X_test[columns],
            categorical_features=[column for column in categorical_columns if column in columns],
            model_config=MODEL_CONFIG,
            validation_config=VALIDATION_CONFIG,
        )
        ablation_rows.append({
            "experiment": experiment_name,
            "feature_group": experiment_name,
            "num_features": len(columns),
            "fold_auc": [float(score) for score in result["fold_scores"]],
            "fold_auc_mean": result["mean_auc"],
            "fold_auc_std": result["std_auc"],
            "oof_auc": result["oof_auc"],
            "delta_oof_auc_vs_e05": result["oof_auc"] - baseline_result["oof_auc"],
            "runtime": result["runtime"],
            "status": "completed",
        })
ablation = pd.DataFrame(ablation_rows)
display(ablation)

In [ ]:
# 11. Optional tuning — không truyền test data vào objective
if CONFIG["run_tuning"]:
    from credit_scoring.modeling.tuning import tune_lightgbm

    SEARCH_SPACE = {
        "learning_rate": {"type": "log_float", "low": 0.005, "high": 0.08},
        "num_leaves": {"type": "int", "low": 16, "high": 96},
        "min_child_samples": {"type": "int", "low": 40, "high": 200},
        "reg_alpha": {"type": "log_float", "low": 1e-3, "high": 2.0},
        "reg_lambda": {"type": "log_float", "low": 0.1, "high": 20.0},
    }
    tuning_result = tune_lightgbm(
        X_train, target, categorical_columns, SEARCH_SPACE,
        {"n_trials": CONFIG["n_trials"], "n_splits": 3, "timeout": 3_600},
    )
    display(tuning_result["trial_dataframe"].sort_values("value", ascending=False).head())
    print(tuning_result["best_params"], tuning_result["best_score"])


## Submission và run metadata

Smoke không tạo submission. Baseline chỉ tạo `submission.csv` khi `create_submission=True`; schema được kiểm tra trước khi ghi. `run_metadata.json` là manifest cuối của lần chạy, liên kết Git commit, dataset path, feature groups, parameters, scores thực tế và artifact paths.

In [ ]:
# 12. Submission với schema chính xác của competition
submission_path = None
if MODE_CONFIG["create_submission"]:
    submission_path = create_home_credit_submission(
        test_ids=test_ids,
        predictions=baseline_result["test_predictions"],
        output_path=OUTPUT_DIR / "submission.csv",
    )
    submission = pd.read_csv(submission_path)
    assert list(submission.columns) == ["SK_ID_CURR", "TARGET"]
    assert len(submission) == len(test_ids)
    print(f"Submission written to: {submission_path}")

if submission_path is not None:
    artifact_paths["submission"] = submission_path
run_metadata_path = (OUTPUT_DIR / "run_metadata.json").resolve()
artifact_paths["run_metadata"] = run_metadata_path
run_metadata = {
    "status": "completed",
    "experiment_name": CONFIG["experiment_name"],
    "run_mode": CONFIG["run_mode"],
    "git_commit": GIT_COMMIT,
    "dataset_path": str(DATA_DIR.resolve()),
    "n_train": len(X_train),
    "n_test": len(X_test),
    "n_features": X_train.shape[1],
    "numeric_feature_count": len(numeric_columns),
    "categorical_feature_count": len(categorical_columns),
    "enabled_feature_groups": enabled_feature_groups,
    "feature_groups": feature_groups,
    "model_parameters": MODEL_CONFIG,
    "validation_parameters": VALIDATION_CONFIG,
    "fold_scores": [float(score) for score in baseline_result["fold_scores"]],
    "mean_auc": float(baseline_result["mean_auc"]),
    "std_auc": float(baseline_result["std_auc"]),
    "oof_auc": float(baseline_result["oof_auc"]),
    "best_iterations": [int(value) for value in baseline_result["best_iterations"]],
    "runtime": float(baseline_result["runtime"]),
    "artifact_paths": {name: str(path) for name, path in artifact_paths.items()},
}
export_json_artifact(run_metadata, run_metadata_path)
print({name: str(path) for name, path in artifact_paths.items()})

## Kết luận experiment

Chỉ ghi `mean_auc`, `oof_auc`, runtime và artifact vào `docs/experiments/experiment_log.md` sau khi notebook đã chạy hoàn tất. Không ghi metric giả hoặc kết luận dựa trên sample mode.